# PROFHiT on the HF hierarchical benchmark datasets (Kaggle GPU runner)

Before running: in the **Settings** panel on the right, set **Accelerator → GPU T4 x2** (or P100) and **Internet → On**.

This clones `math0110/PROFHiT`, trains PROFHiT on `prison`, `tourism`, `police`, and `m5`, and writes one JSON per dataset to `results/`. To run this unattended (survives closing the browser tab), use **Save Version → Save & Run All (Commit)** instead of running cells interactively — the output files will appear under the notebook's Output tab once it finishes.

In [ ]:
!git clone https://github.com/math0110/PROFHiT.git
%cd PROFHiT

In [ ]:
!pip install -q huggingface_hub

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — check notebook Settings > Accelerator")

## Train all four datasets

Settings below (20 pretrain + 40 train epochs, 100 eval samples, 1 seed) are a step up from the CPU "reduced scale" pass we ran locally — GPU should make this fast enough to afford it. Adjust `COMMON_ARGS` if you want to push closer to the paper's full protocol (100–500 epochs, 5 seeds) once you've seen how long one pass takes here.

Each dataset is wrapped in its own `try/except` so a failure on one (e.g. `m5` running out of memory) doesn't lose the results already saved for the others — important for an unattended run.

In [ ]:
import json
import os
import subprocess
import time

DATASETS = ["prison", "tourism", "police", "m5"]

run_summary = {}
for name in DATASETS:
    print(f"\n{'='*60}\nStarting {name}\n{'='*60}")
    t0 = time.time()
    env = {**os.environ, "PROFHIT_DATASET": name}
    try:
        subprocess.run(["python", "train_hf.py"], env=env, check=True)
        run_summary[name] = {"status": "ok", "elapsed_sec": time.time() - t0}
    except subprocess.CalledProcessError as e:
        run_summary[name] = {"status": "failed", "error": str(e), "elapsed_sec": time.time() - t0}
        print(f"!! {name} failed, continuing with remaining datasets")

print("\nRun summary:")
print(json.dumps(run_summary, indent=2))

## Inspect results

In [ ]:
import json
from pathlib import Path

for f in sorted(Path("results").glob("*.json")):
    d = json.load(open(f))
    row = {
        "seed": d["seed"],
        "rmse": round(d["rmse"], 3),
        "mae": round(d["mae"], 3),
        "mape": round(d["mape"], 3),
        "wape": round(d["wape"], 3),
        "crps": round(d["crps"], 3),
        "log_score": round(d["log_score"], 3),
        "calibration_score": round(d["calibration_score"], 3),
        "dce": round(d["dce"], 3),
    }
    print(f.name, "->", row)